
## GPT from scratch in PyTorch


In [1]:

import torch
import numpy as np
import torch.nn as nn

from torch.nn import functional as F


In [3]:

torch.manual_seed(256)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

block_size        = 40      ## N tokens in sequence
batch_size        = 64 
max_iters         = 6000
eval_interval     = 500     
learning_rate     = 0.00035
eval_iters        = 300
vocab_size        = 88  ## 65

## every id for a given token is embedded to vector of this size
n_embd            = 512                  
n_head            = 8         ## 8 attention heads
n_layer           = 6         ## 6 eoncoder layers
dropout           = 0.2


In [4]:

file_path = '/home/vpathaka/ITS_520/Training_VetGPT/corrected_vet_data1.txt'

text = ''

input_file2 = 'corrected_vet_data1.txt'

with open(input_file2, 'r', encoding='utf-8') as f:
    text = f.read()


In [5]:

print("length of data in letter or characters")
len(text)




length of data in letter or characters


1001532

In [6]:

list(set(text))


['9',
 'Z',
 '8',
 'c',
 'k',
 '*',
 'x',
 'N',
 'q',
 'v',
 ' ',
 'l',
 'r',
 'R',
 'g',
 '/',
 '7',
 ']',
 'D',
 'S',
 "'",
 'G',
 '!',
 'E',
 'm',
 '+',
 'd',
 '$',
 '3',
 'e',
 'L',
 'T',
 'V',
 '%',
 'z',
 'n',
 'W',
 '4',
 ')',
 'U',
 '.',
 '0',
 't',
 '2',
 '(',
 'K',
 'b',
 '-',
 'M',
 'o',
 '\n',
 'F',
 'X',
 's',
 'A',
 '?',
 'u',
 '1',
 ',',
 '6',
 '=',
 'B',
 'Y',
 'w',
 '>',
 'P',
 '5',
 'J',
 'C',
 '"',
 'f',
 '[',
 'i',
 'I',
 'y',
 ':',
 '&',
 'p',
 ';',
 'h',
 'a',
 '#',
 'j',
 '_',
 'O',
 'Q',
 'H']

In [7]:

the_chars  = sorted(     list(set(text))     )

vocab_size = len( the_chars )      ## 65

print(  len(the_chars)  )

print(  ''.join(the_chars)  )

## The printed oputput
## !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz



87

 !"#$%&'()*+,-./0123456789:;=>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz


In [8]:

stoi = { ch:i for i, ch in enumerate(the_chars) }
itos = { i:ch for i, ch in enumerate(the_chars) }


In [9]:

print( stoi )
print( itos )


{'\n': 0, ' ': 1, '!': 2, '"': 3, '#': 4, '$': 5, '%': 6, '&': 7, "'": 8, '(': 9, ')': 10, '*': 11, '+': 12, ',': 13, '-': 14, '.': 15, '/': 16, '0': 17, '1': 18, '2': 19, '3': 20, '4': 21, '5': 22, '6': 23, '7': 24, '8': 25, '9': 26, ':': 27, ';': 28, '=': 29, '>': 30, '?': 31, 'A': 32, 'B': 33, 'C': 34, 'D': 35, 'E': 36, 'F': 37, 'G': 38, 'H': 39, 'I': 40, 'J': 41, 'K': 42, 'L': 43, 'M': 44, 'N': 45, 'O': 46, 'P': 47, 'Q': 48, 'R': 49, 'S': 50, 'T': 51, 'U': 52, 'V': 53, 'W': 54, 'X': 55, 'Y': 56, 'Z': 57, '[': 58, ']': 59, '_': 60, 'a': 61, 'b': 62, 'c': 63, 'd': 64, 'e': 65, 'f': 66, 'g': 67, 'h': 68, 'i': 69, 'j': 70, 'k': 71, 'l': 72, 'm': 73, 'n': 74, 'o': 75, 'p': 76, 'q': 77, 'r': 78, 's': 79, 't': 80, 'u': 81, 'v': 82, 'w': 83, 'x': 84, 'y': 85, 'z': 86}
{0: '\n', 1: ' ', 2: '!', 3: '"', 4: '#', 5: '$', 6: '%', 7: '&', 8: "'", 9: '(', 10: ')', 11: '*', 12: '+', 13: ',', 14: '-', 15: '.', 16: '/', 17: '0', 18: '1', 19: '2', 20: '3', 21: '4', 22: '5', 23: '6', 24: '7', 25: '8',

In [10]:

encode = lambda s: [ stoi[c]          for c in s   ] 

encode("Vinay Teja Pathakamuri")


[53,
 69,
 74,
 61,
 85,
 1,
 51,
 65,
 70,
 61,
 1,
 47,
 61,
 80,
 68,
 61,
 71,
 61,
 73,
 81,
 78,
 69]

In [11]:

decode = lambda l: ''.join(   itos[i] for i in l   )    

decode([55,
 73,
 78,
 65,
 89,
 1,
 53,
 69,
 74,
 65,
 1,
 49,
 65,
 84,
 72,
 65,
 75,
 65,
 77,
 85,
 82,
 73])



KeyError: 89

In [12]:

data = torch.tensor(   encode(text), dtype=torch.long   )

print( data )


tensor([39, 61, 72,  ..., 82, 61, 72])


In [13]:

n          = int(   0.9*len(data)   )

train_data = data[:n]
val_data   = data[n:]


In [14]:

def get_batch(split):
    if split == "train":
        data = train_data
    else:
        data = val_data
        
    ix = torch.randint(   len(data) - block_size, (batch_size,)   )
    
    x  = torch.stack(    [  data[   i : i+block_size ]     for i in ix ]    ) 
    y  = torch.stack(    [  data[ i+1 : i+1+block_size ]   for i in ix ]    )
    
    x, y = x.to(device), y.to(device)

    return x, y


In [15]:

temp_batch_size = 4
temp_block_size = 16

## select random starting points for the 4 sentences
ix = torch.randint(   
            len(data) - block_size, 
            (temp_batch_size,)   
)

print( ix )


tensor([347472, 369458, 619859, 969489])


In [16]:

for index_temp in ix:
    print(  data[index_temp]  )



tensor(61)
tensor(61)
tensor(65)
tensor(15)


In [17]:

x  = torch.stack(    
    [ data[   i : i+  temp_block_size ]   for i in ix ] 
    
) 

y  = torch.stack(    
    [ data[ i+1 : i+1+ temp_block_size ]  for i in ix ]    
)

print(x)
print(y)



tensor([[61, 74, 64,  1, 63, 75, 78, 74, 65, 61, 72,  1, 78, 65, 66, 72],
        [61, 70, 75, 78,  1, 79, 75, 81, 78, 63, 65,  1, 75, 66,  1, 41],
        [65, 78, 72, 65, 81, 71, 69, 74, 79, 13,  1, 69, 74, 80, 65, 78],
        [15, 20, 19, 13, 21, 18, 24,  1, 21, 19, 18, 15, 69, 74, 14, 69]])
tensor([[74, 64,  1, 63, 75, 78, 74, 65, 61, 72,  1, 78, 65, 66, 72, 65],
        [70, 75, 78,  1, 79, 75, 81, 78, 63, 65,  1, 75, 66,  1, 41, 74],
        [78, 72, 65, 81, 71, 69, 74, 79, 13,  1, 69, 74, 80, 65, 78, 66],
        [20, 19, 13, 21, 18, 24,  1, 21, 19, 18, 15, 69, 74, 14, 69, 74]])


In [18]:

@torch.no_grad()    ## for efficient processing
def estimate_loss():
    out = {}
    model.eval()   ## set to no training
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()  ## back to training
    return out




## NN Architectures


In [19]:

class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        
        self.key   = nn.Linear(n_embd, head_size, bias=False)  ## [512, 64]
        self.query = nn.Linear(n_embd, head_size, bias=False)  ## [512, 64]
        self.value = nn.Linear(n_embd, head_size, bias=False)  ## [512, 64]

        tril_def = torch.tril( torch.ones(block_size, block_size) )  ## [40, 40]
        
        self.register_buffer(
                  'tril', 
                  tril_def
               )
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        
        B, T, E = x.shape   ## [batch_size, 40, 512]
        
        k = self.key(   x )            ## k = (B, T, 64)
        q = self.query( x )            ## q = (B, T, 64)

        E2 = 64     ## I think this is 64 and not 512
        ## (B, T, E) @ (B, E, T)  -> (B, T, T)
        wei = q @ k.transpose(-2, -1) * E2 ** -0.5        
        
        wei = wei.masked_fill(
                      self.tril[:T, :T] == 0, 
                      float('-inf')
        )   
        
        ## (B, T, T)
        wei = F.softmax( wei, dim= -1 )         ## (B, T, T)
        wei = self.dropout(   wei   )
        
        ## perform weighted aggregation of values
        
        v   = self.value(  x  )   ## x = (B, 40, E)
        out = wei @ v             ## (B, T, T) @ (B, T, 64) -> (B, T, 64)
        
        return out
        


In [20]:


class FeedForward(nn.Module):

    def __init__(self, n_embd):         ## 512
        
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),      ## [512, 4*512]
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),      ## [4*512, 512]
            nn.Dropout(dropout),
        )
        
    def forward(self, x):
        return self.net(x)


In [21]:

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):    ## (8, 64)
        super().__init__()
        self.heads = nn.ModuleList(  [ Head(head_size) for _ in range(num_heads) ] )
        self.proj  = nn.Linear(n_embd, n_embd)   ## 512, 512
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        out = torch.cat(   [ h(x) for h in self.heads ], dim = -1   )
        out = self.proj(  out   )
        out = self.dropout(   out   )
        return out



In [22]:

class Block(nn.Module):
    
    def __init__(self, n_embd, n_head):     ## (512, 8)
        super().__init__()
        head_size = n_embd // n_head        ## 64
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward( n_embd)    ## 512
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)
        
    def forward(self, x):
        x = x + self.sa(     self.ln1(x)      )
        x = x + self.ffwd(   self.ln2(x)      )
        return x


In [23]:

class GPTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)   ## [65, 512]
        self.pos_emb_table = nn.Embedding(block_size, n_embd)     ## [block, 512]
        
        self.blocks = nn.Sequential(
                *[   Block(n_embd, n_head=n_head) for _ in range(n_layer)    ]
        )
        
        self.ln_f    = nn.LayerNorm(  n_embd    )        
        self.lm_ffw_head = nn.Linear(n_embd, vocab_size)  ## [512, 65] # FFW Layer
        
    def forward(self, idx, targets=None):
        B, T = idx.shape     ## (Batch, 40)
        ## ids and targets are both (B, T) tensors of integers
        
        tok_emb = self.token_embedding_table(idx)      
        pos_emb = self.pos_emb_table(torch.arange(T, device=device))  
        
        x = tok_emb + pos_emb    ## [B, T, E] or [64, 40, 512]

        ## This is the architecture
        x = self.blocks(  x  )   ## (B, T, E)        
        x = self.ln_f(    x  )   ## (B, T, E)   ## norm
        logits = self.lm_ffw_head(x)         ## [B, 40, 65] 
        
        if targets is None:
            loss = None
        else:
            B, T, E  = logits.shape
            logits  = logits.view( B*T, E)
            targets = targets.view(B*T)
            loss    = F.cross_entropy(logits, targets)
        return logits, loss
        
    def generate(self, idx, max_new_tokens):    ## idx is (B, T)
        for _ in range(max_new_tokens):
            ## crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)    ## ## get preds
            logits = logits[:, -1, :]    ## focus on last one (B, E)
            probs = F.softmax(logits, dim= -1)    ## (B, E) get probs
            idx_next = torch.multinomial(probs, num_samples=1)     ## (B, 1) selected
            idx = torch.cat(  (idx, idx_next), dim=1  )   ## (B, T+1) append sample to running sequence
        return idx
            


In [24]:

model   = GPTModel()

m       = model.to(device)

optimizer = torch.optim.Adam(  m.parameters(), lr=learning_rate   )



In [25]:


for iter in range(max_iters):
    
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    
    ## eval the loss
    logits, loss = m(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)   ## zero out
    loss.backward()
    optimizer.step()


step 0: train loss 4.6531, val loss 4.6518
step 500: train loss 1.9976, val loss 2.2206
step 1000: train loss 1.7326, val loss 2.0202
step 1500: train loss 1.5989, val loss 1.9266
step 2000: train loss 1.5176, val loss 1.9102
step 2500: train loss 1.4588, val loss 1.8697
step 3000: train loss 1.3992, val loss 1.8516
step 3500: train loss 1.3586, val loss 1.8481
step 4000: train loss 1.3260, val loss 1.8458
step 4500: train loss 1.2799, val loss 1.8456
step 5000: train loss 1.2469, val loss 1.8535
step 5500: train loss 1.2165, val loss 1.8602


In [26]:


## Starting token  id_sos = 0
sos_context = torch.zeros(  (1, 1),  dtype=torch.long, device=device   )   

generated_text = m.generate(sos_context, max_new_tokens=500)[0].tolist()

print(  decode(generated_text)   )




is recessful these of an onenergy elements infections . a legaozoite- calcium wassawetweet, the deficiency . if dogs rickets rated influent in hubls may needs and in bite for stimulation activity accumulation . is a commonly a s
1 struvite of the ure aspiratory drug the mescription .
,,,, rachea, and more retion to gauge: to more develop in agents dogs to secondary arrest test decus or allow food a band ow via casus and in cats; permitaneous worlds and exercisin.
( Tuxzene, DNA)and Weakers KL, a


In [27]:

sos_context = torch.ones(  (1, 1),  dtype=torch.long, device=device   )   

generated_text = m.generate(sos_context, max_new_tokens=500)[0].tolist()

print(  decode(generated_text)   )


 .
a remilds delayed should non-spametherapy .
, associated with hepatic distression katne insufficient pharmacokinetine, C., K. Weins et al. 2001. Asoba- absorption of is treat workely every 8 10 the period. a hight dilatered mastimulates tation. DM can pasture present with of the urethral patients (e.g., how, machoriol). thyroid glawly dies are used to streen.- he penetry mezodges in the cutaneously caused by a high less of concentration level. Ins a cats extraction . Merocainal Internal Med Ve


In [46]:

new_lst = encode("rabies in dogs")


In [47]:

new_np = np.array(  new_lst   )
new_np


array([78, 61, 62, 69, 65, 79,  1, 69, 74,  1, 64, 75, 67, 79])

In [48]:

new_context = torch.tensor(new_np, dtype=torch.long, device=device )


new_context = new_context.view( (1, -1))
new_context 


tensor([[78, 61, 62, 69, 65, 79,  1, 69, 74,  1, 64, 75, 67, 79]],
       device='cuda:0')

In [49]:

generated_text = m.generate(new_context, max_new_tokens=500)[0].tolist()

print(  decode(generated_text)   )


rabies in dogs. With allergens. Anticobacteria is a metabolism is typically, foxilute in caloric drug
is a 3 rare,s, increased with gastrinolamin Bones ment to brain access Diffusion of ear puppy > Focus triglyceride L Uerial Sampl Ala Pa Association 2043;Reseenee, inflam-proved or swocked. cattle.
a cat of Health Aunixls. Journal of the American Veterinary (NRC), muscarili and estromuscular Kindkes JL, Kara J .. vasculture may resteres whether distreamount), 16 body weight 92 contains a consume of the preven



## Figuring out dimensions


In [50]:

new_context.shape


torch.Size([1, 14])

In [51]:

sos_context_tmp = torch.ones(  (1, 1),  dtype=torch.long, device=device   ) 
sos_context_tmp.shape


torch.Size([1, 1])